# Guider Moments — Ensemble Analysis (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-07-16  
**Last Modified:** 2026-07-16  
**Status:** In Progress  
**Keywords:** guiders, second moments, ellipticity, image motion, turbulence, ensemble  

## Description

Plots from the combined per-(expId, detector) guider moment summary dataset
(the Hive-partitioned parquet written by the Snakemake pipeline). Summarizes the
second-moment decomposition across many exposures: moment distributions, the
additivity check (coadd vs static-per-stamp + image-motion), optics-vs-motion
comparisons, and the ensemble-averaged temporal PSDs.

**Input:** `output/<dataset>/moments/` (partitioned by dayObs).
**Based on:** `guider_star_ellipticity.ipynb`, `code/guiderMoments.py`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-16 | Aaron Roodman | Initial ensemble-analysis notebook |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Load](#setup)
3. [Moment Distributions](#dist)
4. [Additivity Check](#additivity)
5. [Optics vs Image Motion](#optics-motion)
6. [Ensemble-Averaged PSD](#psd)
7. [Temporal Fluctuations & Diagnostics](#temporal)
8. [Further Ideas](#ideas)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
dataset = 'night_20260709'                 # dataset name under output/
moments_dir = f'output/{dataset}/moments'  # Hive-partitioned dataset dir
output_dir = f'output/{dataset}/plots'     # where to save figures

save_figs = False                          # set True to write PNGs to output_dir

# PSD frequency band edges (must match the pipeline; guiderMoments.PSD_BIN_EDGES)
psd_bin_edges = [0.03, 0.1, 0.2, 0.5, 1.0, 2.5]  # Hz

<a id='setup'></a>
## Setup & Load

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.dataset as pds

pd.set_option('display.max_columns', None)
if save_figs:
    os.makedirs(output_dir, exist_ok=True)

def savefig(fig, name):
    if save_figs:
        fig.savefig(os.path.join(output_dir, f'guider_{name}_{dataset}.png'), dpi=110)

In [ ]:
# Read the partitioned dataset (dayObs restored from the folder names).
if os.path.isdir(moments_dir):
    df = pds.dataset(moments_dir, format='parquet', partitioning='hive').to_table().to_pandas()
else:
    df = pd.read_parquet(moments_dir)   # single-file fallback

print(f'{len(df)} rows | {df.expId.nunique()} exposures | {df.detector.nunique()} sensors | nights {sorted(df.dayObs.unique())}')
df.head()

In [ ]:
# Frequency band centers/widths for the binned PSDs.
edges = np.asarray(psd_bin_edges, float)
band_ctr = np.sqrt(edges[:-1] * edges[1:])   # geometric center per band
band_w = np.diff(edges)
n_band = len(band_ctr)

def stack_psd(col):
    """Stack a list-valued PSD column into an (n_rows, n_band) array."""
    out = np.full((len(df), n_band), np.nan)
    for i, v in enumerate(df[col].to_numpy()):
        if v is not None and len(v) == n_band:
            out[i] = np.asarray(v, float)
    return out

def yx_line(ax, x, y, **kw):
    lo = np.nanmin([np.nanmin(x), np.nanmin(y)])
    hi = np.nanmax([np.nanmax(x), np.nanmax(y)])
    pad = 0.05 * (hi - lo if hi > lo else 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], 'k--', lw=1, **kw)
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect('equal')

<a id='dist'></a>
## Moment Distributions

Distributions of the traceless shape moments $Q_1,Q_2$ and the size moment $T$
(arcsec$^2$), overlaid for the three kinds: `coadd`, `stamp` (static per-stamp
mean = optics bound), and `motion` (image-motion covariance = turbulence).

In [ ]:
kinds = ['coadd', 'stamp', 'motion']
colors = {'coadd': 'k', 'stamp': 'C0', 'motion': 'C3'}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, q in zip(axes, ['Q1', 'Q2', 'T']):
    vals = np.concatenate([df[f'{q}_{k}'].dropna().to_numpy() for k in kinds])
    lo, hi = np.nanpercentile(vals, [1, 99])
    bins = np.linspace(lo, hi, 60)
    for k in kinds:
        ax.hist(df[f'{q}_{k}'], bins=bins, histtype='step', lw=1.8,
                color=colors[k], label=k)
    ax.set_xlabel(f'{q} [arcsec$^2$]'); ax.set_yscale('log')
    ax.legend()
axes[0].set_ylabel('sensor-exposures')
fig.suptitle(f'Moment distributions  {dataset}')
fig.tight_layout(); savefig(fig, 'moment_dist')

<a id='additivity'></a>
## Additivity Check

Does the mean of the per-stamp moments plus the centroid variance equal the coadd?
$M^{\rm coadd}\stackrel{?}{=}\langle M^{\rm stamp}\rangle + M^{\rm motion}$, per
$Q_1,Q_2,T$. Points on the dashed $y=x$ line satisfy the identity. The 4th panel is
the fractional residual $(\,{\rm coadd}-[{\rm stamp}+{\rm motion}]\,)/{\rm coadd}$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, q in zip(axes.flat[:3], ['Q1', 'Q2', 'T']):
    x = df[f'{q}_stamp'] + df[f'{q}_motion']
    y = df[f'{q}_coadd']
    ax.scatter(x, y, s=8, alpha=0.3, color='C0')
    yx_line(ax, x, y)
    ax.set_xlabel(f'{q}: stamp + motion [arcsec$^2$]')
    ax.set_ylabel(f'{q}: coadd [arcsec$^2$]'); ax.grid(alpha=0.3)
ax = axes.flat[3]
for q, c in [('Q1', 'C0'), ('Q2', 'C1'), ('T', 'C2')]:
    frac = (df[f'resid_{q}'] / df[f'{q}_coadd']).replace([np.inf, -np.inf], np.nan).dropna()
    lim = np.nanpercentile(np.abs(frac), 98) if len(frac) else 1.0
    ax.hist(frac, bins=np.linspace(-lim, lim, 50), histtype='step', lw=1.8, color=c, label=q)
ax.axvline(0, color='k', lw=1); ax.set_xlabel('fractional residual'); ax.legend()
ax.set_title('coadd - (stamp+motion), / coadd')
fig.suptitle(f'Additivity check  {dataset}')
fig.tight_layout(); savefig(fig, 'additivity')

<a id='optics-motion'></a>
## Optics vs Image Motion

How large is the image-motion (turbulence) contribution relative to the static
per-stamp (optics-bound) shape? Each panel plots the static term against the motion
term on equal axes; points far below the $y=x$ line mean the static (optics) piece
dominates that component. The last panel uses the scalar shape
$Q=\sqrt{Q_1^2+Q_2^2}$.

In [ ]:
df['Qmag_stamp'] = np.hypot(df['Q1_stamp'], df['Q2_stamp'])
df['Qmag_motion'] = np.hypot(df['Q1_motion'], df['Q2_motion'])

panels = [('Q1_stamp', 'Q1_motion', 'Q1'),
          ('Q2_stamp', 'Q2_motion', 'Q2'),
          ('T_stamp', 'T_motion', 'T'),
          ('Qmag_stamp', 'Qmag_motion', '|Q| = sqrt(Q1^2+Q2^2)')]
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, (xs, ys, lab) in zip(axes.flat, panels):
    x, y = df[xs], df[ys]
    ax.scatter(x, y, s=8, alpha=0.3, color='C4')
    yx_line(ax, x, y)
    ax.set_xlabel(f'{lab}: static per-stamp [arcsec$^2$]')
    ax.set_ylabel(f'{lab}: image motion [arcsec$^2$]'); ax.grid(alpha=0.3)
fig.suptitle(f'Static (optics) vs image motion (turbulence)  {dataset}')
fig.tight_layout(); savefig(fig, 'optics_vs_motion')

In [ ]:
# Ratio distributions: how big is motion relative to static, per component.
fig, ax = plt.subplots(figsize=(7, 4))
for a, b, lab, c in [('Qmag_motion', 'Qmag_stamp', '|Q|', 'C4'),
                     ('T_motion', 'T_stamp', 'T', 'C2')]:
    r = (df[a] / df[b]).replace([np.inf, -np.inf], np.nan).dropna()
    r = r[(r > 0) & (r < np.nanpercentile(r, 99))]
    ax.hist(r, bins=50, histtype='step', lw=1.8, color=c, label=f'{lab}: motion/static')
ax.axvline(1.0, color='k', ls='--', lw=1)
ax.set_xlabel('motion / static'); ax.set_ylabel('sensor-exposures'); ax.legend()
ax.set_title(f'Relative size of image motion vs static shape  {dataset}')
fig.tight_layout(); savefig(fig, 'motion_static_ratio')

<a id='psd'></a>
## Ensemble PSD (per-visit variation)

Band-integrated variance per frequency band. First as **violins** over visits (one
distribution per band, showing how the PSD varies image to image), then as the
**individual-visit PSDs overlaid**. Each visit's PSD is the mean band power over its
guide sensors. Centroid motion (`dx,dy`) is in arcsec$^2$; shape/size (`Q1,Q2,T`) in
arcsec$^4$.

In [ ]:
quantities = [('psd_dx', 'arcsec$^2$'), ('psd_dy', 'arcsec$^2$'),
              ('psd_Q1', 'arcsec$^4$'), ('psd_Q2', 'arcsec$^4$'), ('psd_T', 'arcsec$^4$')]

def per_visit_psd(col):
    """(n_visits, n_band) band powers, averaged over sensors within each visit."""
    arr = stack_psd(col)
    with np.errstate(invalid='ignore'):
        return np.array([np.nanmean(arr[rows], axis=0)
                         for rows in df.groupby('expId').indices.values()])

def violin_psd(ax, pv, unit):
    data, pos = [], []
    for b in range(n_band):
        v = pv[:, b]; v = v[np.isfinite(v) & (v > 0)]
        if v.size >= 2:
            data.append(np.log10(v)); pos.append(b)
    if data:
        ax.violinplot(data, positions=pos, showmedians=True, widths=0.8)
    ax.set_xticks(range(n_band))
    ax.set_xticklabels([f'{c:.2f}' for c in band_ctr], fontsize=8)
    ax.set_xlabel('band center f [Hz]'); ax.set_ylabel(f'log10 power [{unit}]')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, unit) in zip(axes.flat, quantities):
    violin_psd(ax, per_visit_psd(col), unit)
    ax.set_title(col); ax.grid(alpha=0.3)
axes.flat[-1].axis('off')
fig.suptitle(f'Per-visit PSD distributions (violin over visits)  {dataset}')
fig.tight_layout(); savefig(fig, 'psd_violin')

In [ ]:
# Individual-visit PSDs overlaid (thin lines) with the ensemble median.
def overlay_psd(ax, pv):
    for row in pv:
        m = np.isfinite(row) & (row > 0)
        if m.sum() >= 2:
            ax.loglog(band_ctr[m], row[m], color='C0', alpha=0.06, lw=0.8)
    med = np.nanmedian(pv, axis=0)
    m = np.isfinite(med) & (med > 0)
    ax.loglog(band_ctr[m], med[m], 'k-o', lw=2, label='median')
    ax.set_xlabel('f [Hz]'); ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, unit) in zip(axes.flat, quantities):
    overlay_psd(ax, per_visit_psd(col))
    ax.set_title(col); ax.set_ylabel(f'band power [{unit}]')
axes.flat[-1].axis('off')
fig.suptitle(f'Individual-visit PSDs overlaid  {dataset}')
fig.tight_layout(); savefig(fig, 'psd_overlay')

<a id='temporal'></a>
## Temporal Fluctuations & Diagnostics

The temporal covariance of $(Q_1,Q_2,T)$ probes higher-order wavefront fluctuations
($T\!\sim\!Z4$ defocus; $Q\!\sim\!Z4\times Z5/Z6$). Plus the static/optics fraction,
coherence times, and a cross-check of the moment ellipticity against the tracker's
HSM ellipticity.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 9))

# (a) temporal variance distributions (turbulence higher-order signal)
for col, c in [('var_T', 'C2'), ('var_Q1', 'C0'), ('var_Q2', 'C1')]:
    v = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    v = v[v > 0]
    if len(v):
        ax[0, 0].hist(np.log10(v), bins=50, histtype='step', lw=1.8, color=c, label=col)
ax[0, 0].set_xlabel('log10 temporal variance [arcsec$^4$]'); ax[0, 0].legend()
ax[0, 0].set_title('temporal (Q1,Q2,T) variance')

# (b) static (optics) fraction
for col, c in [('frac_Q1_static', 'C0'), ('frac_Q2_static', 'C1')]:
    f = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    ax[0, 1].hist(f, bins=np.linspace(-0.5, 1.5, 41), histtype='step', lw=1.8, color=c, label=col)
ax[0, 1].axvline(0.5, color='grey', ls='--'); ax[0, 1].legend()
ax[0, 1].set_xlabel('static / (static + motion)'); ax[0, 1].set_title('optics fraction of shape')

# (c) coherence times
for col, c in [('tau_x', 'C0'), ('tau_y', 'C1')]:
    t = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    t = t[(t > 0) & (t < np.nanpercentile(t, 99))]
    ax[1, 0].hist(t, bins=40, histtype='step', lw=1.8, color=c, label=col)
ax[1, 0].set_xlabel('integral timescale [s]'); ax[1, 0].legend()
ax[1, 0].set_title('centroid coherence time')

# (d) moment ellipticity vs HSM ellipticity (cross-check)
e1_mom = df['Q1_coadd'] / df['T_coadd']
ax[1, 1].scatter(df['e1_med'], e1_mom, s=8, alpha=0.3, color='C3')
yx_line(ax[1, 1], df['e1_med'], e1_mom)
ax[1, 1].set_xlabel('e1 (HSM per-stamp median)'); ax[1, 1].set_ylabel('e1 = Q1_coadd / T_coadd')
ax[1, 1].set_title('moment vs HSM ellipticity'); ax[1, 1].grid(alpha=0.3)
fig.suptitle(f'Temporal fluctuations & diagnostics  {dataset}')
fig.tight_layout(); savefig(fig, 'temporal')

In [ ]:
# Does defocus fluctuation (var_T) track the tip/tilt jitter?
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
m = (df['var_T'] > 0) & (df['jitter_altaz'] > 0)
ax[0].scatter(df.loc[m, 'jitter_altaz'], np.sqrt(df.loc[m, 'var_T']), s=8, alpha=0.3, color='C2')
ax[0].set_xlabel('Alt/Az jitter [arcsec]'); ax[0].set_ylabel('sqrt(var_T) [arcsec$^2$]')
ax[0].set_title('defocus fluctuation vs tip/tilt jitter'); ax[0].grid(alpha=0.3)
# size vs seeing proxy
ax[1].scatter(df['fwhm_med'], df['T_coadd'], s=8, alpha=0.3, color='C5')
ax[1].set_xlabel('median FWHM [arcsec]'); ax[1].set_ylabel('T_coadd [arcsec$^2$]')
ax[1].set_title('coadd size vs FWHM'); ax[1].grid(alpha=0.3)
fig.tight_layout(); savefig(fig, 'correlations')

<a id='ideas'></a>
## Further Ideas

Other plots worth adding as the ensemble grows:

- **Focal-plane maps**: median $Q_1,Q_2$ whiskers per sensor across the field (optics
  pattern) vs the motion whiskers (turbulence) — the `run_guider_whiskers.py` view,
  but split by static/motion and aggregated over the night.
- **Trends vs time / airmass**: `T_coadd`, jitter, `var_T` vs seqNum, elapsed night
  time, or `sec(z)` (from `alt`) — turbulence should scale with airmass/seeing.
- **Per-sensor systematics**: box/violin of each moment by `detector` — reveals a
  static optical pattern (corner-dependent astigmatism) distinct from turbulence.
- **Cross-covariance sign**: `cov_Q1T`, `cov_Q2T` (defocus-astigmatism coupling) vs
  rotator angle `camRotAngle` — tests the $Z4\times Z5/Z6$ picture.
- **Motion anisotropy**: orientation of the centroid covariance (`Q1_motion,Q2_motion`)
  vs wind/Alt-Az direction — dome-seeing / wind signature.
- **Default vs moment comparison**: the `gm_*` GuiderMetricsBuilder slopes/scatter vs
  the moment-based quantities, to validate against the standard pipeline.
- **Noise-floor subtraction**: subtract `psd_*_floor * band_w` from the band powers
  and the temporal variances before interpreting the turbulence amplitude.